In [15]:
import sys
from pathlib import Path
import torch

ROOT = Path(r"C:\Users\gagan\Desktop\Federated Learning\federated-xray-classification")

sys.path.insert(0, str(ROOT))

In [18]:
from src.models.cbam_cnn import (
    CBAMCNN
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cuda


In [19]:
model = CBAMCNN()

model.load_state_dict(
    torch.load(
        "../models/federated_cbam_best.pth",
        map_location=device
    )
)

model.to(device)

model.eval()

C:\Users\gagan\AppData\Local\Temp\ipykernel_38548\2965872004.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(


CBAMCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU(inplace=True)
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (7): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (8): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (9): ReLU(inplace=True)
    (10): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (11): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (12): ReLU(inplace=True)
    (13): MaxPool2d(kernel_size=2, stride=2, padding=0, dilatio

In [20]:
all_labels = []
all_predictions = []
all_probabilities = []

In [22]:
from src.preprocessing.dataset_loader import build_dataset_index
from src.preprocessing.dataloaders import (
    create_datasets,
    create_dataloaders
)
from src.preprocessing.splitter import create_stratified_split

In [26]:
from pathlib import Path

ROOT = Path(
    r"C:\Users\gagan\Desktop\Federated Learning\federated-xray-classification"
)

train_dir = ROOT / "data" / "raw" / "train"
test_dir = ROOT / "data" / "raw" / "test"

train_paths, train_labels = build_dataset_index(train_dir)
test_paths, test_labels = build_dataset_index(test_dir)

print(len(train_paths), len(train_labels))

5216 5216


In [27]:
print("Number of train images:", len(train_paths))
print("Number of train labels:", len(train_labels))

Number of train images: 5216
Number of train labels: 5216


In [28]:
(
    X_train,
    X_val,
    y_train,
    y_val
) = create_stratified_split(
    train_paths,
    train_labels
)

In [29]:
(
    train_dataset,
    val_dataset,
    test_dataset
) = create_datasets(
    X_train,
    y_train,
    X_val,
    y_val,
    test_paths,
    test_labels
)

In [30]:
(
    train_loader,
    val_loader,
    test_loader
) = create_dataloaders(
    train_dataset,
    val_dataset,
    test_dataset,
    batch_size=32
)

In [31]:
print(len(test_loader.dataset))

624


In [32]:
with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)

        outputs = model(images)

        probabilities = torch.softmax(
            outputs,
            dim=1
        )[:,1]

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        all_labels.extend(
            labels.numpy()
        )

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_probabilities.extend(
            probabilities.cpu().numpy()
        )

0.006489981897175312 0.9983893632888794
0.4942048192024231 0.9999066591262817
0.004607288166880608 0.9981623291969299
0.4869997203350067 0.999936580657959
0.003783692140132189 0.9988603591918945
0.4261815547943115 0.9997828602790833
0.0043410565704107285 0.9991218447685242
0.47100406885147095 0.9999091625213623
0.006854364648461342 0.9991389513015747
0.49277162551879883 0.9997978806495667
0.007780035492032766 0.9975499510765076
0.5001935362815857 0.9998683929443359
0.006505551747977734 0.9979192614555359
0.4989875257015228 0.9999974966049194
0.005275988485664129 0.9995761513710022
0.4831734001636505 0.9998053908348083
0.002974345115944743 0.9996236562728882
0.513604462146759 0.9995447993278503
0.0020846067927777767 0.9997414946556091
0.5125645995140076 0.9998487234115601
0.004582608584314585 0.999488353729248
0.5106358528137207 0.9996141195297241
0.004433942027390003 0.9993870258331299
0.4917420446872711 0.9991791844367981
0.0015647702384740114 0.9998179078102112
0.5086601972579956 0.9

In [33]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

results = {

    "accuracy":
        accuracy_score(
            all_labels,
            all_predictions
        ),

    "precision":
        precision_score(
            all_labels,
            all_predictions
        ),

    "recall":
        recall_score(
            all_labels,
            all_predictions
        ),

    "f1_score":
        f1_score(
            all_labels,
            all_predictions
        ),

    "roc_auc":
        roc_auc_score(
            all_labels,
            all_probabilities
        )
}

In [34]:
print("\n=== FEDERATED TEST RESULTS ===\n")

for key, value in results.items():

    print(
        f"{key}: {value:.4f}"
    )


=== FEDERATED TEST RESULTS ===

accuracy: 0.7724
precision: 0.7756
recall: 0.8949
f1_score: 0.8310
roc_auc: 0.8577


In [35]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    all_labels,
    all_predictions
)

print(cm)

[[133 101]
 [ 41 349]]


In [36]:
from sklearn.metrics import classification_report

print(

    classification_report(
        all_labels,
        all_predictions
    )

)

              precision    recall  f1-score   support

           0       0.76      0.57      0.65       234
           1       0.78      0.89      0.83       390

    accuracy                           0.77       624
   macro avg       0.77      0.73      0.74       624
weighted avg       0.77      0.77      0.76       624



In [38]:
import json

ROOT = Path(
    r"C:\Users\gagan\Desktop\Federated Learning\federated-xray-classification"
)

with open(
    ROOT / "results" / "metrics" / "federated_results.json",
    "w"
) as f:

    json.dump(
        results,
        f,
        indent=4
    )